# Verify the image issues and build clean manifests

The audit (01) showed the **labels are clean** (9,408 rows, 96 study eyes, 87 patients, 2 visits × 49 B-scans, 100 % matched to images). Three things on the **image side** still need checking before any model is trained:

1. **Non-study eyes.** Every patient folder has both `OD` and `OS`, but only one is the study eye (96 labelled eyes vs 190 on disk). The other eye is excluded.
2. **Extra files.** Almost every volume has ≠ 49 numbered B-scans, and some images are 768×768 RGB (probably the near-IR fundus image) next to the 504×496 B-scans. This step finds out exactly what the extra files are.
3. **Duplicate volumes.** 187 pairs of volumes had the same quick fingerprint, 183 of them across different patients. That would mean leakage between patients if real, so it is re-checked here with a stricter comparison.

Then it writes the tables every later notebook uses:

| File | One row per | Used for |
| --- | --- | --- |
| `manifests/volumes.csv` | study-eye visit | visit order, BCVA/CST, exclusions |
| `manifests/labelled_bscans.csv` | labelled B-scan (first and last visit) | training and testing |
| `manifests/unlabelled_bscans.csv` | B-scan of an intermediate visit | label-shift estimation (RQ3) |
| `manifests/folds.csv` | patient | 5-fold split grouped by patient |

**Run:** `Restart & Run All`. Needs `audit_outputs/` from notebook 01. Reads images only to check them; writes to `manifests/`.

In [1]:
from pathlib import Path
import os, re, hashlib, datetime
import numpy as np
import pandas as pd
from PIL import Image

PROJECT = Path(os.environ.get('OLIVES_PROJECT', '/Users/parvezali/Research Proposal'))
AUDIT   = PROJECT / 'audit_outputs'
OUT     = PROJECT / 'manifests'
OUT.mkdir(exist_ok=True)
SEED, N_FOLDS = 0, 5

TARGETS = {'IRF': 'Fluid (IRF)', 'DRT_ME': 'DRT/ME', 'IRHRF': 'IR HRF',
           'FAVF': 'Fully attached vitreous face', 'PAVF': 'Partially attached vitreous face'}

img  = pd.read_csv(AUDIT / 'image_index.csv')
bio  = pd.read_csv(AUDIT / 'biomarker_labels_clean.csv')
clin = pd.read_csv(AUDIT / 'clinical_by_visit.csv') if (AUDIT / 'clinical_by_visit.csv').exists() else None
print(f'images on disk: {len(img):,} | label rows: {len(bio):,} | clinical visits: {0 if clin is None else len(clin):,}')

NOTES = []
def note(s):
    NOTES.append(s); print('[NOTE]', s)

images on disk: 159,281 | label rows: 9,408 | clinical visits: 1,594


## 1 Keep the study eyes only
The study eye is the one with biomarker labels. For TREX this also equals the eye letter in the folder name (`0201GOD` → OD). The patient ID comes from the label file (`Patient_ID`), because TREX gives the two eyes of one patient different folder names (`0201GOD`, `0201TOS` → patient 201). Splits must group by this ID.

In [2]:
eyes = (bio.drop_duplicates('eye_key')
           .set_index('eye_key')[['study', 'cohort', 'patient', 'eye', 'Patient_ID', 'Eye_ID', 'first_lab', 'last_lab']])
eyes['patient_id'] = eyes.study + '-' + eyes.Patient_ID.astype(int).astype(str)
print(eyes.groupby('study').agg(eyes=('Eye_ID', 'size'), patients=('patient_id', 'nunique')))
print('patients with both eyes in the study:', int((eyes.groupby('patient_id').size() > 1).sum()))

trex = eyes[eyes.study == 'TREX']
assert (trex.patient.str[-2:] == trex.eye).all(), 'TREX study eye does not match the folder name'

on_disk = img.eye_key.unique()
fellow = sorted(set(on_disk) - set(eyes.index))
note(f'{len(fellow)} non-study (fellow) eyes on disk are excluded from all analyses')
img = img[img.eye_key.isin(eyes.index)].copy()
print(f'study-eye images: {len(img):,} in {img.vol_key.nunique():,} volumes')

       eyes  patients
study                
PRIME    40        40
TREX     56        47
patients with both eyes in the study: 9
[NOTE] 94 non-study (fellow) eyes on disk are excluded from all analyses
study-eye images: 81,322 in 1,593 volumes


## 2 What are the extra files in each volume?
Reads only the image header (fast). A file counts as a B-scan if it has a number in its name **and** has the most common B-scan size.

In [3]:
def header(p):
    try:
        with Image.open(p) as im:
            return im.width, im.height, im.mode
    except Exception:
        return np.nan, np.nan, 'unreadable'

hdr = [header(p) for p in img.path]
img['w'], img['h'], img['mode'] = zip(*hdr)
img['size'] = img.w.astype('Int64').astype(str) + 'x' + img.h.astype('Int64').astype(str)
print('Image sizes / modes in study-eye folders:')
display(img.groupby(['size', 'mode']).size().rename('files'))

numbered = img.scan_idx.notna()
BSCAN_SIZE = img.loc[numbered, 'size'].mode().iat[0]
img['is_bscan'] = numbered & (img['size'] == BSCAN_SIZE)
print('B-scan size:', BSCAN_SIZE)

extra = img[~img.is_bscan]
print(f'non-B-scan files: {len(extra):,}')
if len(extra):
    display(extra.assign(name=extra.file).groupby(['size', 'mode']).name.agg(['size', lambda s: sorted(set(s))[:5]])
                 .rename(columns={'size': 'files', '<lambda_0>': 'example names'}))

per_vol = img[img.is_bscan].groupby('vol_key').agg(n_bscans=('path', 'size'), min_idx=('scan_idx', 'min'), max_idx=('scan_idx', 'max'))
per_vol['contiguous'] = (per_vol.max_idx - per_vol.min_idx + 1) == per_vol.n_bscans
print('B-scans per volume after removing extras:')
display(per_vol.n_bscans.value_counts().sort_index().rename('volumes'))
print('volumes with gaps in scan numbers:', int((~per_vol.contiguous).sum()))
if (per_vol.n_bscans != 49).any():
    note(f'{int((per_vol.n_bscans != 49).sum())} study-eye volumes do not have 49 B-scans after removing extras')
else:
    note('every study-eye volume has exactly 49 B-scans once the non-B-scan files are removed')

Image sizes / modes in study-eye folders:


size     mode
504x496  L       78135
         RGB         1
768x768  L        1593
         RGB      1593
Name: files, dtype: int64

B-scan size: 504x496
non-B-scan files: 3,186


files                                      example names
size    mode                                                          
768x768 L      1593  [fundus_OD_V1.tif, fundus_OD_V10.tif, fundus_O...
        RGB    1593  [TREXA_000004_001.tif, TREXA_001.tif, TREXB_00...

B-scans per volume after removing extras:


n_bscans
36       1
46       1
48       1
49    1588
97       2
Name: volumes, dtype: int64

volumes with gaps in scan numbers: 3
[NOTE] 5 study-eye volumes do not have 49 B-scans after removing extras


## 3 Re-check duplicate volumes
A stricter fingerprint: the bytes of the B-scans at positions 0, 12, 24, 36 and 48 (B-scans only, extras ignored). Any volumes that still match are then compared on **all** of their B-scans. Blank frames (almost no pixel variation) are reported, since identical blank frames would make different volumes look the same.

In [4]:
def md5(p):
    return hashlib.md5(Path(p).read_bytes()).hexdigest()

bs = img[img.is_bscan].sort_values(['vol_key', 'scan_idx'])
fp = {}
for vk, g in bs.groupby('vol_key'):
    ps = g.path.tolist()
    picks = [ps[int(round(i))] for i in np.linspace(0, len(ps) - 1, 5)]
    fp[vk] = '|'.join(md5(p) for p in picks)
vol_fp = pd.Series(fp, name='fp5')
cand = vol_fp[vol_fp.duplicated(keep=False)]
print(f'volumes sharing the 5-scan fingerprint: {len(cand)} in {cand.nunique()} groups')

def full_hash(vk):
    return [md5(p) for p in bs.loc[bs.vol_key == vk, 'path']]

def is_blank(p):
    with Image.open(p) as im:
        return float(np.asarray(im.convert('L'), dtype=np.float32).std()) < 2.0

dup_rows = []
for gid, (f, grp) in enumerate(cand.groupby(cand)):
    vks = grp.index.tolist()
    hashes = {vk: full_hash(vk) for vk in vks}
    ref = hashes[vks[0]]
    identical = all(h == ref for h in hashes.values())
    shared = sum(len(set(h)) for h in hashes.values()) - len(set().union(*map(set, hashes.values())))
    blank_share = np.mean([is_blank(p) for p in bs.loc[bs.vol_key == vks[0], 'path'].iloc[[0, 24, -1]]])
    pats = {vk.split('|')[1] for vk in vks}
    ek = {vk.rsplit('|', 1)[0] for vk in vks}
    kind = 'same eye, different visits' if len(ek) == 1 else ('same patient, both eyes' if len(pats) == 1 else 'different patients')
    for vk in vks:
        dup_rows.append(dict(dup_group=gid, vol_key=vk, kind=kind, all_49_identical=identical, blank_frames=blank_share > 0.5))
dups = pd.DataFrame(dup_rows, columns=['dup_group', 'vol_key', 'kind', 'all_49_identical', 'blank_frames'])
if len(dups):
    display(dups.groupby(['kind', 'all_49_identical', 'blank_frames']).dup_group.nunique().rename('groups'))
    dups.to_csv(OUT / 'duplicate_volumes_verified.csv', index=False)
    real = dups[dups.all_49_identical & ~dups.blank_frames]
    labelled = set(bio.vol_key)
    note(f'{real.dup_group.nunique()} groups of fully identical, non-blank study-eye volumes '
         f'involving {real.vol_key.isin(labelled).sum()} labelled volumes; groups by kind: {real.groupby("kind").dup_group.nunique().to_dict()}')
else:
    note('no duplicated study-eye volumes once only B-scans are compared: the earlier flag came from the extra files')

volumes sharing the 5-scan fingerprint: 4 in 2 groups


kind                        all_49_identical  blank_frames
same eye, different visits  True              False           2
Name: groups, dtype: int64

[NOTE] 2 groups of fully identical, non-blank study-eye volumes involving 0 labelled volumes; groups by kind: {'same eye, different visits': 2}


## 4 Volume manifest, exclusions and folds
Exclusion rules (fixed now, before any result is seen):
- **duplicate**: the later volume of a fully identical same-eye pair, and every volume of an identical pair across eyes or patients (cannot tell which is genuine)
- **incomplete**: fewer than 49 B-scans
- **short follow-up** is *not* excluded but flagged (`short_followup`): PRIME < 52 weeks or TREX < 10 visits between labelled visits; used as a sensitivity analysis.

In [5]:
vol = (img[img.is_bscan].groupby(['eye_key', 'vol_key', 'visit', 'visit_num'])
          .agg(n_bscans=('path', 'size')).reset_index()
          .sort_values(['eye_key', 'visit_num']))
vol = vol.join(eyes[['study', 'cohort', 'eye', 'patient_id', 'Eye_ID', 'first_lab', 'last_lab']], on='eye_key')
vol['visit_order'] = vol.groupby('eye_key').cumcount() + 1
vol['n_visits']    = vol.groupby('eye_key').vol_key.transform('size')
vol['timepoint']   = np.select([vol.visit_num == vol.first_lab, vol.visit_num == vol.last_lab], ['first', 'last'], 'middle')
vol.loc[vol.visit_num < vol.first_lab, 'timepoint'] = 'before_first'
vol.loc[vol.visit_num > vol.last_lab,  'timepoint'] = 'after_last'
span = vol.last_lab - vol.first_lab
vol['followup']       = span
vol['short_followup'] = np.where(vol.study == 'PRIME', span < 52, span < 9)

if clin is not None:
    vol = vol.merge(clin[['vol_key', 'BCVA', 'CST']], on='vol_key', how='left')

vol['exclude'] = ''
if len(dups):
    real = dups[dups.all_49_identical & ~dups.blank_frames].merge(vol[['vol_key', 'visit_num']], on='vol_key')
    for _, g in real.groupby('dup_group'):
        drop = g.sort_values('visit_num').vol_key.iloc[1:] if g.kind.iat[0] == 'same eye, different visits' else g.vol_key
        vol.loc[vol.vol_key.isin(drop), 'exclude'] = 'duplicate'
vol.loc[(vol.n_bscans < 49) & (vol.exclude == ''), 'exclude'] = 'incomplete'

print(vol.timepoint.value_counts())
print(vol.exclude.replace('', 'kept').value_counts())
bad_lab = vol[vol.timepoint.isin(['first', 'last']) & (vol.exclude != '')]
if len(bad_lab):
    note(f'{len(bad_lab)} labelled volumes would be excluded: {bad_lab.vol_key.tolist()}; their eyes are dropped from the main analysis')
note(f'{int(vol.drop_duplicates("eye_key").short_followup.sum())} eyes have short follow-up (flag only)')
vol.to_csv(OUT / 'volumes.csv', index=False)

# folds: grouped by patient, balanced by trial
pats = eyes.drop_duplicates('patient_id')[['patient_id', 'study']].sort_values('patient_id')
rng = np.random.default_rng(SEED)
folds = []
for s, g in pats.groupby('study'):
    ids = g.patient_id.to_numpy().copy(); rng.shuffle(ids)
    folds += [dict(patient_id=p, study=s, fold=i % N_FOLDS) for i, p in enumerate(ids)]
folds = pd.DataFrame(folds)
folds.to_csv(OUT / 'folds.csv', index=False)
fe = eyes.reset_index().merge(folds[['patient_id', 'fold']], on='patient_id')
display(fe.groupby(['fold', 'study']).size().unstack(fill_value=0).assign(total=lambda d: d.sum(axis=1)))

timepoint
middle    1401
first       96
last        96
Name: count, dtype: int64
exclude
kept          1588
incomplete       3
duplicate        2
Name: count, dtype: int64
[NOTE] 11 eyes have short follow-up (flag only)


study,PRIME,TREX,total
fold,,,
0,8,12,20
1,8,12,20
2,8,9,17
3,8,11,19
4,8,12,20


## 5 B-scan manifests
`labelled_bscans.csv`: first- and last-visit B-scans with the five study biomarkers (empty cells stay empty and are skipped per biomarker). `unlabelled_bscans.csv`: B-scans from the visits in between, for label-shift estimation.

In [6]:
excluded_eyes = set(vol.loc[vol.timepoint.isin(['first', 'last']) & (vol.exclude != ''), 'eye_key'])
keep_cols = ['img_path', 'eye_key', 'vol_key', 'study', 'timepoint', 'scan_idx'] + list(TARGETS.values())
lab = bio[keep_cols].rename(columns={v: k for k, v in TARGETS.items()}).rename(columns={'img_path': 'path'})
lab = lab.merge(eyes[['patient_id']], left_on='eye_key', right_index=True).merge(folds[['patient_id', 'fold']], on='patient_id')
lab = lab[~lab.eye_key.isin(excluded_eyes)]
lab.to_csv(OUT / 'labelled_bscans.csv', index=False)

mid_vols = vol[(vol.timepoint == 'middle') & (vol.exclude == '') & ~vol.eye_key.isin(excluded_eyes)]
unl = img[img.is_bscan & img.vol_key.isin(mid_vols.vol_key)][['path', 'eye_key', 'vol_key', 'study', 'visit', 'scan_idx']]
unl = unl.merge(eyes[['patient_id']], left_on='eye_key', right_index=True).merge(folds[['patient_id', 'fold']], on='patient_id')
unl.to_csv(OUT / 'unlabelled_bscans.csv', index=False)

print(f'labelled B-scans: {len(lab):,} from {lab.eye_key.nunique()} eyes / {lab.patient_id.nunique()} patients')
print(f'unlabelled B-scans: {len(unl):,} from {unl.vol_key.nunique():,} intermediate visits')
print('Missing labels per biomarker (skipped for that biomarker only):')
display(lab[list(TARGETS)].isna().sum().rename('empty cells'))

labelled B-scans: 9,408 from 96 eyes / 87 patients
unlabelled B-scans: 68,500 from 1,396 intermediate visits
Missing labels per biomarker (skipped for that biomarker only):


IRF       0
DRT_ME    0
IRHRF     0
FAVF      3
PAVF      6
Name: empty cells, dtype: int64

## 6 Summary

In [7]:
lines = [f'# Clean manifests: summary\n\nGenerated {datetime.date.today()}.\n',
         f'- Study eyes: {eyes.shape[0]} ({(eyes.study == "PRIME").sum()} PRIME, {(eyes.study == "TREX").sum()} TREX), patients: {eyes.patient_id.nunique()}',
         f'- B-scan size: {BSCAN_SIZE}',
         f'- Labelled B-scans kept: {len(lab):,} ({lab.eye_key.nunique()} eyes)',
         f'- Unlabelled intermediate B-scans: {len(unl):,} ({unl.vol_key.nunique():,} visits)',
         '', '## Notes', ''] + [f'- {n}' for n in NOTES]
(OUT / 'manifest_summary.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
print('\nFiles:', sorted(p.name for p in OUT.iterdir()))

# Clean manifests: summary

Generated 2026-10-25.

- Study eyes: 96 (40 PRIME, 56 TREX), patients: 87
- B-scan size: 504x496
- Labelled B-scans kept: 9,408 (96 eyes)
- Unlabelled intermediate B-scans: 68,500 (1,396 visits)

## Notes

- 94 non-study (fellow) eyes on disk are excluded from all analyses
- 5 study-eye volumes do not have 49 B-scans after removing extras
- 2 groups of fully identical, non-blank study-eye volumes involving 0 labelled volumes; groups by kind: {'same eye, different visits': 2}
- 11 eyes have short follow-up (flag only)

Files: ['duplicate_volumes_verified.csv', 'folds.csv', 'labelled_bscans.csv', 'manifest_summary.md', 'unlabelled_bscans.csv', 'volumes.csv']
